# Verification notebook for Kissing number in $d=11$

This notebook is a self-contained mathematical and computational companion to the Kissing number in $d=11$ section in the paper.

The task template, including the prompt given to the agents, is available [here](https://github.com/dualverse-ai/station/tree/main/example/research_alpha_evolve/kissing_margin).

The outline is as follows:

- [1. Problem Definitions](#/notebooks/kissing_number/verification.ipynb/section-1)
- [2. S1. Three exact $604$-point kissing configurations.](#/notebooks/kissing_number/verification.ipynb/section-2)
- [3. S2. An algebraic construction for a $604$-point kissing configuration in $\mathbb R^{11}$.](#/notebooks/kissing_number/verification.ipynb/section-3)
- [4. S3. Why the classical $D_{11}$ construction stops at $582$.](#/notebooks/kissing_number/verification.ipynb/section-4) ([Formal proof](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/kissing_number/formal_proofs/s3))
- [5. Additional findings](#/notebooks/kissing_number/verification.ipynb/section-5)

The required packages are:

- `NumPy`
- `SymPy`
- `IPython`

<a id="section-1"></a>

## 1. Problem Definitions

The kissing number $K(d)$ is the largest number of nonoverlapping unit spheres that can touch a central unit sphere in $\mathbb R^d$. Equivalently, it is the largest size of a set $X\subset S^{d-1}$ such that

$$
\langle x,y\rangle\le\frac12\qquad(x\ne y\in X).
$$

We scale every vector to squared norm $4$. The same condition then becomes

$$
\langle x,y\rangle\le2.
$$

A pair with inner product $2$ is a **contact**. Contact count, the multiset of pairwise inner products, and the number of antipodal pairs are preserved by orthogonal transformations and relabeling. A mismatch in any one of these invariants proves that two configurations are non-isometric.

The certificates encode every coordinate as

$$
x_k=\frac{P_k+Q_k\sqrt2}{6},\qquad P_k,Q_k\in\mathbb Z.
$$

These are shared routines used by all cells below. The next cell authenticates the certificate bundle and defines exact arithmetic in $\mathbb Q(\sqrt2)$, Gram-matrix calculations, and row-set comparison.

In [1]:
from __future__ import annotations

from collections import Counter, deque
from fractions import Fraction
from hashlib import sha256
from itertools import combinations, product
from pathlib import Path

import numpy as np
import sympy as sp
from IPython.display import Markdown, display

DATA_PATH = Path("kissing_certificates.npz")
EXPECTED_SHA256 = "61b3a572ad194a3b97d8bcae8694d57ae676328a81ef0e4db95e57a2c9d44f37"
actual_hash = sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_hash == EXPECTED_SHA256, (actual_hash, EXPECTED_SHA256)

bundle = np.load(DATA_PATH, allow_pickle=False)
expected_keys = {
    "config_coefficients", "config_labels", "config_denominator", "core_size",
    "d12_lift_3", "shell_582",
}
assert set(bundle.files) == expected_keys

coefficients = bundle["config_coefficients"].astype(np.int64)
labels = bundle["config_labels"].tolist()
DEN = int(bundle["config_denominator"])
CORE_N = int(bundle["core_size"])
P_all, Q_all = coefficients[..., 0], coefficients[..., 1]
assert coefficients.shape == (3, 604, 11, 2)
assert labels == ["1", "2", "3"] and DEN == 6 and CORE_N == 496

def q2_sign(a: int | Fraction, b: int | Fraction) -> int:
    # Exact sign of a+b*sqrt(2), for rational a,b.
    a, b = Fraction(a), Fraction(b)
    if a == 0 and b == 0:
        return 0
    if b == 0:
        return 1 if a > 0 else -1
    if a == 0:
        return 1 if b > 0 else -1
    if (a > 0 and b > 0) or (a < 0 and b < 0):
        return 1 if a > 0 else -1
    comparison = a * a - 2 * b * b
    if a > 0:  # a>0>b
        return 1 if comparison > 0 else -1
    return 1 if comparison < 0 else -1  # b>0>a

def q2_add(x, y):
    return (x[0] + y[0], x[1] + y[1])

def q2_mul(x, y):
    return (x[0] * y[0] + 2 * x[1] * y[1], x[0] * y[1] + x[1] * y[0])

def q2_scale(c, x):
    c = Fraction(c)
    return (c * x[0], c * x[1])

def gram_coefficients(P, Q):
    P, Q = np.asarray(P, dtype=np.int64), np.asarray(Q, dtype=np.int64)
    return P @ P.T + 2 * (Q @ Q.T), P @ Q.T + Q @ P.T

def row_keys(P, Q):
    return {tuple(np.concatenate([p, q]).tolist()) for p, q in zip(P, Q)}

print("certificate SHA-256:", actual_hash)
print("arrays:", ", ".join(sorted(bundle.files)))


certificate SHA-256: 61b3a572ad194a3b97d8bcae8694d57ae676328a81ef0e4db95e57a2c9d44f37
arrays: config_coefficients, config_denominator, config_labels, core_size, d12_lift_3, shell_582


<a id="section-2"></a>

## 2. S1. Three exact $604$-point kissing configurations.

### 2.1 Three isometry classes

**Theorem 2.1 (exact finite configurations).**

There exist three configurations, labelled Constructions 1, 2, and 3, of $604$ norm-$2$ vectors in $\mathbb R^{11}$ with all pairwise inner products at most $2$.  Their exact invariants are:

| invariant | 1 | 2 | 3 |
|---|---:|---:|---:|
| contacts $\langle x_i,x_j\rangle=2$ | 19,704 | 22,904 | 22,840 |
| antipodal pairs | 302 | 302 | 238 |
| distinct off-diagonal inner products | 22 | 14 | 15 |

Consequently the three configurations are pairwise non-isometric.

**Verification.** The next cell checks distinctness, norm, every kissing inequality, contacts, antipodes, angle counts, and the shared block architecture of the three certificates.


In [2]:
def audit_configuration(P, Q, denominator=DEN):
    A, B = gram_coefficients(P, Q)
    norm_target = 4 * denominator * denominator
    assert np.all(np.diag(A) == norm_target) and np.all(np.diag(B) == 0)
    assert len(row_keys(P, Q)) == len(P)

    violations = 0
    contacts = 0
    palette = set()
    antipodes = 0
    keys = row_keys(P, Q)
    for i in range(len(P)):
        antipodes += tuple(np.concatenate([-P[i], -Q[i]]).tolist()) in keys
        for j in range(i + 1, len(P)):
            a, b = int(A[i, j]), int(B[i, j])
            palette.add((Fraction(a, denominator**2), Fraction(b, denominator**2)))
            contacts += (a == 2 * denominator**2 and b == 0)
            violations += q2_sign(a - 2 * denominator**2, b) > 0
    assert violations == 0 and antipodes % 2 == 0
    return {
        "N": len(P), "contacts": contacts, "antipodal_pairs": antipodes // 2,
        "angles": len(palette), "violations": violations,
    }

audits = {label: audit_configuration(P, Q) for label, P, Q in zip(labels, P_all, Q_all)}
expected = {
    "1": {"N": 604, "contacts": 19704, "antipodal_pairs": 302, "angles": 22, "violations": 0},
    "2": {"N": 604, "contacts": 22904, "antipodal_pairs": 302, "angles": 14, "violations": 0},
    "3": {"N": 604, "contacts": 22840, "antipodal_pairs": 238, "angles": 15, "violations": 0},
}
assert audits == expected
display(Markdown("|configuration|N|contacts|antipodal pairs|angles|violations|\n|---|---:|---:|---:|---:|---:|\n" +
    "\n".join(f"|{k}|{v['N']}|{v['contacts']:,}|{v['antipodal_pairs']}|{v['angles']}|{v['violations']}|" for k,v in audits.items())))

# Exact block architecture in the common coordinate frame.
core_1, core_2, core_3 = (row_keys(P[:CORE_N], Q[:CORE_N]) for P, Q in zip(P_all, Q_all))
extension_2 = row_keys(P_all[1, CORE_N:], Q_all[1, CORE_N:])
extension_3 = row_keys(P_all[2, CORE_N:], Q_all[2, CORE_N:])
assert core_1 == core_2
assert len(core_2 & core_3) == 432
assert len(core_2 - core_3) == len(core_3 - core_2) == 64
assert extension_2 == extension_3 and len(extension_2) == 108
print("exact shared architecture: 432 shared core + 64 phase core + 108-point extension")


|configuration|N|contacts|antipodal pairs|angles|violations|
|---|---:|---:|---:|---:|---:|
|1|604|19,704|302|22|0|
|2|604|22,904|302|14|0|
|3|604|22,840|238|15|0|

exact shared architecture: 432 shared core + 64 phase core + 108-point extension


### 2.2 An explicit algebraic construction of Construction 1

Use zero-based coordinate indices to match the construction below.  Split the coordinates into

$$
E=\{0,1,2,3,4,5,7,8\},\qquad F=\{6,9,10\}.
$$

The integer core consists of the sixteen axes $\{\pm2e_i:i\in E\}$ and all sixteen signings of each of the thirty four-support blocks encoded below.  Thus the core has

$$
16+30\cdot16=496
$$

rows.  On the residual three-space use the rational rotation

$$
R=\frac13
\begin{pmatrix}
1&2&2\\
2&1&-2\\
-2&2&-1
\end{pmatrix},
\qquad R^{\mathsf T}R=I_3.
$$

Four coordinate pairs in $E$ form the matching

$$
\{0,4\},\quad\{1,7\},\quad\{2,3\},\quad\{5,8\}.
$$

For every pair, take its four signed roots $\pm e_i\pm e_j$.  Adjoining each such root to the six-point residual cross $\sqrt2R\{\pm f_1,\pm f_2,\pm f_3\}$ produces $4\cdot4\cdot6=96$ rows.  Finally, embed the twelve cuboctahedron vertices

$$
\sqrt2R\{(\pm1,\pm1,0),(\pm1,0,\pm1),(0,\pm1,\pm1)\}
$$

in $F$.  The complete construction therefore has

$$
496+96+12=604
$$

rows.

Once the four-pair matching is stated, the construction requires no computer search.  Among the $112$ signed $D_8$ roots, exactly the sixteen roots supported on the four pairs above have all six rotated-cross rows compatible with the generated core.

**Proposition 2.2 (explicit construction of Construction 1).**

The construction gives Construction 1: $604$ distinct norm-$2$ vectors satisfying the kissing inequalities.

**Verification.** The next cell constructs every row from the displayed ingredients, filters the $112$ signed $D_8$ roots, and checks every kissing inequality. It then compares the resulting row set with the bundled certificate to identify the construction with Construction 1.

In [3]:
# Thirty four-support blocks for the 480 signed weight-four core rows.
CONSTRUCTION_1_SUPPORTS_NATIVE = [
    (0,1,2,6), (0,1,3,9), (0,1,4,7), (0,1,8,10), (0,2,3,4),
    (0,2,5,9), (0,2,7,10), (0,3,5,10), (0,3,6,8), (0,4,5,8),
    (0,5,6,7), (0,7,8,9), (1,2,3,7), (1,2,5,10), (1,2,8,9),
    (1,3,4,10), (1,3,5,6), (1,4,5,9), (1,4,6,8), (1,5,7,8),
    (2,3,5,8), (2,4,5,6), (2,4,7,9), (2,4,8,10), (2,6,7,8),
    (3,4,6,7), (3,4,8,9), (3,5,7,9), (3,7,8,10), (4,5,7,10),
]
CONSTRUCTION_1_SNAP_DIMS = (0,1,2,3,4,5,7,8)
CONSTRUCTION_1_RESIDUAL_DIMS = (6,9,10)
CONSTRUCTION_1_PORT_MATCHING = ((0,4), (1,7), (2,3), (5,8))
CONSTRUCTION_1_ROTATION_NUMERATOR = np.asarray(
    [[1,2,2], [2,1,-2], [-2,2,-1]], dtype=np.int64
)
CONSTRUCTION_1_NATIVE_TO_COMMON = (0,9,10,4,3,2,8,5,7,1,6)

assert len(set(CONSTRUCTION_1_SUPPORTS_NATIVE)) == 30
assert max(len(set(s) & set(t)) for s,t in combinations(CONSTRUCTION_1_SUPPORTS_NATIVE,2)) <= 2
assert np.array_equal(
    CONSTRUCTION_1_ROTATION_NUMERATOR.T @ CONSTRUCTION_1_ROTATION_NUMERATOR,
    9 * np.eye(3, dtype=np.int64),
)

# Work throughout in the notebook's denominator-six encoding.
constructed_core_p = []
for axis in CONSTRUCTION_1_SNAP_DIMS:
    for sign in (-1,1):
        row = np.zeros(11, dtype=np.int64)
        row[axis] = 12 * sign
        constructed_core_p.append(row)
for support in CONSTRUCTION_1_SUPPORTS_NATIVE:
    for signs in product((-1,1), repeat=4):
        row = np.zeros(11, dtype=np.int64)
        for axis, sign in zip(support, signs):
            row[axis] = 6 * sign
        constructed_core_p.append(row)
constructed_core_p = np.asarray(constructed_core_p, dtype=np.int64)
constructed_core_q = np.zeros_like(constructed_core_p)
assert constructed_core_p.shape == (496,11)

def construction_1_cross_module(port):
    # Six rows a + sqrt(2) R(+-f_j), in denominator-six pair form.
    p_rows, q_rows = [], []
    for residual_axis in range(3):
        for sign in (-1,1):
            basis = np.zeros(3, dtype=np.int64)
            basis[residual_axis] = sign
            p_rows.append(6 * port)
            q = np.zeros(11, dtype=np.int64)
            q[list(CONSTRUCTION_1_RESIDUAL_DIMS)] = 2 * (CONSTRUCTION_1_ROTATION_NUMERATOR @ basis)
            q_rows.append(q)
    return np.asarray(p_rows), np.asarray(q_rows)

def all_cross_inner_products_le_two(P1,Q1,P2,Q2):
    A = P1 @ P2.T + 2 * (Q1 @ Q2.T)
    B = P1 @ Q2.T + Q1 @ P2.T
    return all(
        q2_sign(int(a) - 2 * DEN**2, int(b)) <= 0
        for a,b in zip(A.ravel(), B.ravel())
    )

# Derive the sixteen ports from the generated core, starting from all 112 D8 roots.
all_d8_roots = []
for i,j in combinations(CONSTRUCTION_1_SNAP_DIMS,2):
    for si,sj in product((-1,1), repeat=2):
        port = np.zeros(11, dtype=np.int64)
        port[i], port[j] = si, sj
        all_d8_roots.append(port)

selected_ports = []
for port in all_d8_roots:
    module_p, module_q = construction_1_cross_module(port)
    if all_cross_inner_products_le_two(
        module_p, module_q, constructed_core_p, constructed_core_q
    ):
        selected_ports.append(port)

expected_ports = set()
for i,j in CONSTRUCTION_1_PORT_MATCHING:
    for si,sj in product((-1,1), repeat=2):
        port = np.zeros(11, dtype=np.int64)
        port[i], port[j] = si, sj
        expected_ports.add(tuple(port.tolist()))
assert len(all_d8_roots) == 112
assert {tuple(p.tolist()) for p in selected_ports} == expected_ports
assert len(selected_ports) == 16

household_p, household_q = [], []
for port in selected_ports:
    module_p, module_q = construction_1_cross_module(port)
    household_p.extend(module_p)
    household_q.extend(module_q)
household_p = np.asarray(household_p, dtype=np.int64)
household_q = np.asarray(household_q, dtype=np.int64)
assert household_p.shape == household_q.shape == (96,11)

cuboctahedron = set()
for a,b in product((-1,1), repeat=2):
    cuboctahedron.update({(a,b,0), (a,0,b), (0,a,b)})
hub_p, hub_q = [], []
for vertex in sorted(cuboctahedron):
    p = np.zeros(11, dtype=np.int64)
    q = np.zeros(11, dtype=np.int64)
    q[list(CONSTRUCTION_1_RESIDUAL_DIMS)] = 2 * (
        CONSTRUCTION_1_ROTATION_NUMERATOR @ np.asarray(vertex, dtype=np.int64)
    )
    hub_p.append(p)
    hub_q.append(q)
hub_p, hub_q = np.asarray(hub_p), np.asarray(hub_q)
assert hub_p.shape == hub_q.shape == (12,11)

P_1_constructed_native = np.vstack([constructed_core_p, household_p, hub_p])
Q_1_constructed_native = np.vstack([constructed_core_q, household_q, hub_q])
assert P_1_constructed_native.shape == Q_1_constructed_native.shape == (604,11)
assert len(row_keys(P_1_constructed_native,Q_1_constructed_native)) == 604

# This coordinate permutation is only for comparison with Constructions 2 and 3 in their common frame.
P_1_constructed = np.zeros_like(P_1_constructed_native)
Q_1_constructed = np.zeros_like(Q_1_constructed_native)
for old,new in enumerate(CONSTRUCTION_1_NATIVE_TO_COMMON):
    P_1_constructed[:,new] = P_1_constructed_native[:,old]
    Q_1_constructed[:,new] = Q_1_constructed_native[:,old]

construction_1_matches_certificate = (
    row_keys(P_1_constructed,Q_1_constructed) == row_keys(P_all[0],Q_all[0])
)
construction_1_audit = audit_configuration(P_1_constructed,Q_1_constructed)
assert construction_1_matches_certificate
assert construction_1_audit == expected["1"]
assert row_keys(P_1_constructed[:496],Q_1_constructed[:496]) == core_1

print("generated core: 16 axes + 30 supports x 16 signs =", len(constructed_core_p))
print("D8-root filter:", len(all_d8_roots), "candidates ->", len(selected_ports), "ports")
print("generated tail: 16 ports x 6 cross rows + 12 hub rows =", len(household_p)+len(hub_p))
print("exact row-set match with the bundled certificate:", construction_1_matches_certificate)
print("generated invariants:", construction_1_audit)


generated core: 16 axes + 30 supports x 16 signs = 496
D8-root filter: 112 candidates -> 16 ports
generated tail: 16 ports x 6 cross rows + 12 hub rows = 108
exact row-set match with the bundled certificate: True
generated invariants: {'N': 604, 'contacts': 19704, 'antipodal_pairs': 302, 'angles': 22, 'violations': 0}


The generated Construction 1 is the EinsteinArena class.

**Station Reference.** Construction 1 comes from S2 Archive #51 and Evaluations #1355, #1368, and #1377. Its direct core-and-port derivation is supported by S2 Evaluations #2257, #2274, #2287, and #2290 and Archive #104/Evaluation #2310. Construction 3 comes from S1 Archives #73 and #74 and Evaluations #1115, #1118, #1125, and #1147. Construction 2 comes from S1 Archives #120 and #121 and Evaluations #1432, #1438, #1440, #1906, and #1917.

**Related Work.** [Ganzhinov (2022)](https://arxiv.org/abs/2207.08266) gave the earlier $592$-point lower bound, and [Georgiev et al. (2025)](https://arxiv.org/abs/2511.02864) reported AlphaEvolve's $593$-point construction. The Station and EinsteinArena discoveries were concurrent and independent. The [EinsteinArena paper](https://arxiv.org/abs/2606.10402), first submitted on June 9, 2026, reported $604$ points; its [exact coordinates](https://github.com/togethercomputer/EinsteinArena-new-SOTA/blob/main/kissing-number/solutions/solution_n%3D604_d%3D11.json) define Construction 1 here. Thus Construction 1 is an independent Station rediscovery, whereas the distinct contact counts of Constructions 2 and 3 establish two additional, apparently novel isometry classes. Construction 3 was first made public by Station on June 14, 2026, with an explicit construction and proof in [this notebook](https://github.com/dualverse-ai/station/blob/88f052eb1f4a16eb100315a6173852bb1226bcdb/news/2026-06-14-kissing.ipynb).

<a id="section-3"></a>

## 3. S2. An algebraic construction for a $604$-point kissing configuration in $\mathbb R^{11}$.

### 3.1 Construction 3

Construction 3 consists of a $496$-point integer core and a $108$-point extension.  The core contains sixteen coordinate-axis vectors, all sixteen signings of twenty-two four-coordinate supports, and eight prescribed signings of another sixteen supports.  Thus its size is

$$
16+22\cdot16+16\cdot8=496.
$$

For the extension, rotate the coordinate plane spanned by $e_6,e_8$ through $45^\circ$ and use the orthonormal frame

$$
f_6=\frac{e_6+e_8}{\sqrt2},\qquad
f_8=\frac{e_6-e_8}{\sqrt2},
$$

with $f_i=e_i$ on the other coordinates.  Eleven signed patterns in this frame generate $54$ line representatives.  Taking both orientations produces the $108$-point extension.  The complete support and sign lists are encoded directly below.

**Proposition 3.1 (explicit construction of Construction 3).**

The construction gives Construction 3: $604$ distinct norm-$2$ vectors satisfying the kissing inequalities.

**Verification.** The next cell constructs every row in denominator-six quadratic-pair form and checks all norms and pairwise kissing inequalities. It then compares the unordered row set with the bundled certificate and verifies the decomposition into $496$ core points and $54$ antipodal extension lines.


In [4]:
CONFIG3_AXIS_COORDS = (0, 2, 3, 4, 5, 7, 9, 10)
CONFIG3_FULL_SUPPORTS = [
    (0,1,2,10), (0,1,4,9), (0,1,5,7), (0,2,3,7),
    (0,2,4,6), (0,3,4,10), (0,3,5,9), (0,5,6,10),
    (0,6,7,9), (1,2,3,9), (1,2,4,5), (1,3,4,7),
    (1,3,5,10), (1,7,9,10), (2,3,5,6), (2,4,7,10),
    (2,5,7,9), (2,6,9,10), (3,4,6,9), (3,6,7,10),
    (4,5,6,7), (4,5,9,10),
]
CONFIG3_HALF_SUPPORTS = [
    ((0,2,5,8), -1), ((0,2,8,9), 1), ((0,4,5,8), 1),
    ((0,4,7,8), -1), ((0,7,8,10), 1), ((0,8,9,10), -1),
    ((2,3,4,8), 1), ((2,3,8,10), -1), ((2,4,8,9), -1),
    ((2,5,8,10), 1), ((3,4,5,8), -1), ((3,5,7,8), 1),
    ((3,7,8,9), -1), ((3,8,9,10), 1), ((4,7,8,9), 1),
    ((5,7,8,10), -1),
]

# Generate the 496-point integer core in denominator-six pair form.
core_3_p = []
for axis in CONFIG3_AXIS_COORDS:
    for sign in (-1, 1):
        row = np.zeros(11, dtype=np.int64)
        row[axis] = 12 * sign
        core_3_p.append(row)
for support in CONFIG3_FULL_SUPPORTS:
    for signs in product((-1, 1), repeat=4):
        row = np.zeros(11, dtype=np.int64)
        for axis, sign in zip(support, signs):
            row[axis] = 6 * sign
        core_3_p.append(row)
for support, required_sign_at_8 in CONFIG3_HALF_SUPPORTS:
    position_8 = support.index(8)
    for signs in product((-1, 1), repeat=4):
        if signs[position_8] != required_sign_at_8:
            continue
        row = np.zeros(11, dtype=np.int64)
        for axis, sign in zip(support, signs):
            row[axis] = 6 * sign
        core_3_p.append(row)
core_3_p = np.asarray(core_3_p, dtype=np.int64)
core_3_q = np.zeros_like(core_3_p)
assert core_3_p.shape == (496, 11)

def config3_row(p_entries=(), q_entries=()):
    p = np.zeros(11, dtype=np.int64)
    q = np.zeros(11, dtype=np.int64)
    for axis, value in p_entries:
        p[axis] = value
    for axis, value in q_entries:
        q[axis] = value
    return p, q

# Eleven signed families for one representative of each extension line.
extension_3_blocks = {name: [] for name in "ABCDEFGHIJK"}
for e, h, t in product((-1, 1), repeat=3):
    extension_3_blocks["A"].append(config3_row(
        ((0,-6),(1,6*e),(3,6*h)), ((6,3*t),(8,3*t))))
for e, h in product((-1, 1), repeat=2):
    extension_3_blocks["B"].append(config3_row(
        ((0,-6),(3,6*e),(6,6*h),(8,-6*h))))
extension_3_blocks["C"].append(config3_row(((1,-12),)))
for e, h, t in product((-1, 1), repeat=3):
    extension_3_blocks["D"].append(config3_row(
        ((1,-6),(2,6*e),(7,6*t)), ((6,3*h),(8,3*h))))
    extension_3_blocks["E"].append(config3_row(
        ((1,-6),(4,6*e),(10,6*t)), ((6,3*h),(8,3*h))))
    extension_3_blocks["F"].append(config3_row(
        ((1,-6),(5,6*e),(9,6*t)), ((6,3*h),(8,3*h))))
for e, h in product((-1, 1), repeat=2):
    extension_3_blocks["G"].append(config3_row(
        ((1,-6),(6,6*h),(8,-6*h)), ((6,3*e),(8,3*e))))
    extension_3_blocks["H"].append(config3_row(
        ((2,-6),(7,6*e),(6,6*h),(8,-6*h))))
    extension_3_blocks["I"].append(config3_row(
        ((4,-6),(6,6*e),(8,-6*e),(10,6*h))))
    extension_3_blocks["J"].append(config3_row(
        ((5,-6),(6,6*e),(8,-6*e),(9,6*h))))
extension_3_blocks["K"].append(config3_row((), ((6,-6),(8,-6))))

config3_block_counts = {name: len(rows) for name, rows in extension_3_blocks.items()}
assert config3_block_counts == {
    "A":8, "B":4, "C":1, "D":8, "E":8, "F":8,
    "G":4, "H":4, "I":4, "J":4, "K":1,
}
config3_line_pairs = [row for name in "ABCDEFGHIJK" for row in extension_3_blocks[name]]
config3_line_p = np.asarray([p for p, _q in config3_line_pairs], dtype=np.int64)
config3_line_q = np.asarray([q for _p, q in config3_line_pairs], dtype=np.int64)
assert config3_line_p.shape == config3_line_q.shape == (54, 11)

P_3_constructed = np.vstack([core_3_p, config3_line_p, -config3_line_p])
Q_3_constructed = np.vstack([core_3_q, config3_line_q, -config3_line_q])
assert P_3_constructed.shape == Q_3_constructed.shape == (604, 11)
assert len(row_keys(P_3_constructed, Q_3_constructed)) == 604

construction_3_matches_certificate = (
    row_keys(P_3_constructed, Q_3_constructed) == row_keys(P_all[2], Q_all[2])
)
construction_3_audit = audit_configuration(P_3_constructed, Q_3_constructed)
assert construction_3_matches_certificate
assert construction_3_audit == expected["3"]
assert row_keys(core_3_p, core_3_q) == core_3

# The following S2 checks use the equivalent denominator-two encoding.
assert np.all(P_all[2] % 3 == 0) and np.all(Q_all[2] % 3 == 0)
P_3_denom2 = (P_all[2] // 3).astype(np.int64)
Q_3_denom2 = (Q_all[2] // 3).astype(np.int64)

print("generated core: 16 axes + 22 supports x 16 signs + 16 supports x 8 signs =", len(core_3_p))
print("generated extension lines by family:", config3_block_counts)
print("generated extension points: 2 x", len(config3_line_p), "=", 2 * len(config3_line_p))
print("exact row-set match with the bundled certificate:", construction_3_matches_certificate)
print("generated invariants:", construction_3_audit)


generated core: 16 axes + 22 supports x 16 signs + 16 supports x 8 signs = 496
generated extension lines by family: {'A': 8, 'B': 4, 'C': 1, 'D': 8, 'E': 8, 'F': 8, 'G': 4, 'H': 4, 'I': 4, 'J': 4, 'K': 1}
generated extension points: 2 x 54 = 108
exact row-set match with the bundled certificate: True
generated invariants: {'N': 604, 'contacts': 22840, 'antipodal_pairs': 238, 'angles': 15, 'violations': 0}


### 3.2 The extension is built from regular rank-five modules

Use one-based soft coordinates $\{2,7,9\}$ and stiff coordinates $\{1,3,4,5,6,8,10,11\}$.  Construction 3's $54$ extension lines split as

$$
54=6+4\cdot12:
$$

six lines live entirely in the soft three-space, while each of four disjoint stiff-coordinate pairs supports a twelve-line module.  Each module spans dimension five and, on its $24$ oriented rays, has frame eigenvalues

$$
24,24,16,16,16.
$$

Among all $28$ stiff-pair templates, two modules coexist exactly when their stiff pairs are disjoint; hence at most four occur, and a perfect matching attains four.

The next cell identifies the four modules, checks their ranks, frame spectra, and inner-product counts, then tests the compatibility rule for all $28$ stiff-coordinate pairs.

In [5]:
soft = {1,6,8}
selected_pairs = [(0,3),(2,7),(4,10),(5,9)]
cavP, cavQ = P_3_denom2[CORE_N:], Q_3_denom2[CORE_N:]

groups = {"soft": []}
for pair in selected_pairs: groups[pair] = []
for idx,(p,q) in enumerate(zip(cavP,cavQ)):
    outside = tuple(sorted(i for i in range(11) if i not in soft and (p[i] or q[i])))
    if not outside: groups["soft"].append(idx)
    else: groups[outside].append(idx)
assert len(groups["soft"]) == 12 and all(len(groups[pair]) == 24 for pair in selected_pairs)

for pair in selected_pairs:
    idx = groups[pair]
    p,q = cavP[idx],cavQ[idx]
    frame_A = p.T@p + 2*q.T@q
    frame_B = p.T@q + q.T@p
    assert np.all(frame_B==0)
    eig = sorted([int(v) for v,m in sp.Matrix(frame_A//4).eigenvals().items() for _ in range(m) if v])
    assert eig == [16,16,16,24,24]
    ga,gb=gram_coefficients(p,q)
    hist=Counter()
    for i in range(24):
        for j in range(24):
            if i!=j:
                assert gb[i,j]==0
                hist[int(ga[i,j]//4)]+=1
    assert hist==Counter({0:240,-2:144,2:144,-4:24})
module_eigenvalues = eig

# Generate all 28 modules from the first exact template.
stiff = sorted(set(range(11))-soft)
base_pair=selected_pairs[0]
base_idx=groups[base_pair]
bp,bq=cavP[base_idx],cavQ[base_idx]
templates={}
for a,b in combinations(stiff,2):
    p=np.zeros_like(bp);q=np.zeros_like(bq)
    for s in soft: p[:,s],q[:,s]=bp[:,s],bq[:,s]
    p[:,a],q[:,a]=bp[:,base_pair[0]],bq[:,base_pair[0]]
    p[:,b],q[:,b]=bp[:,base_pair[1]],bq[:,base_pair[1]]
    templates[(a,b)]=(p,q)

softP,softQ=cavP[groups["soft"]],cavQ[groups["soft"]]
for p,q in templates.values():
    A=p@p.T+2*q@q.T;B=p@q.T+q@p.T
    for i in range(24):
        for j in range(i+1,24): assert q2_sign(int(A[i,j])-8,int(B[i,j]))<=0
    A=p@softP.T+2*q@softQ.T;B=p@softQ.T+q@softP.T
    assert all(q2_sign(int(a)-8,int(b))<=0 for a,b in zip(A.ravel(),B.ravel()))

def modules_compatible(m1,m2):
    p1,q1=templates[m1];p2,q2=templates[m2]
    A=p1@p2.T+2*q1@q2.T;B=p1@q2.T+q1@p2.T
    return all(q2_sign(int(a)-8,int(b))<=0 for a,b in zip(A.ravel(),B.ravel()))

sharing=disjoint=0
for e,f in combinations(templates,2):
    ok=modules_compatible(e,f)
    if set(e)&set(f): sharing+=1; assert not ok
    else: disjoint+=1; assert ok
assert (sharing,disjoint)==(168,210)
print("extension rows: 12 soft + 4 x 24 module rays")
print("module rank/frame eigenvalues:", len(module_eigenvalues), module_eigenvalues)
print("all 28 templates: 168 sharing-axis pairs conflict; 210 disjoint pairs coexist")


extension rows: 12 soft + 4 x 24 module rays
module rank/frame eigenvalues: 5 [16, 16, 16, 24, 24]
all 28 templates: 168 sharing-axis pairs conflict; 210 disjoint pairs coexist


### 3.3 The quadratic field is forced by the gluing

**Proposition 3.2 (where irrational Gram entries occur).**

For Construction 3, the core-core and extension-extension Gram blocks are rational. Exactly $18{,}944$ unordered Gram entries are irrational, and every one lies across the core-extension interface.

**Verification.** The next cell writes every normalized Gram entry uniquely as $a+b\sqrt2$ with $a,b\in\mathbb Q$. It verifies that $b=0$ in the core-core and extension-extension blocks and counts exactly $18{,}944$ cross-block entries with $b\ne0$.

**Theorem 3.3 (the extension Gram matrix has no rational realization in $\mathbb R^{11}$).**

The extension Gram matrix has rank $11$ and cannot be realized by vectors with rational coordinates in $\mathbb R^{11}$.

**Verification of rank.** The next code cell exhibits an $11\times11$ principal minor of the normalized extension Gram matrix with determinant $1/512$, proving that its rank is $11$.

**Proof of the rational obstruction.** If the same Gram matrix had rational coordinates in $\mathbb R^{11}$, that minor would be $BB^{\mathsf T}$ for a rational square matrix $B$. Its determinant would then be $(\det B)^2$, a square in $\mathbb Q$, whereas $1/512=2^{-9}$ is not a rational square. $\square$

**Theorem 3.4 (the scalar shear is forced).**

Replace each coordinate $(p+q\sqrt2)/2$ by $(p+qt)/2$ and normalize each resulting nonzero row to norm $2$. The resulting $604$-vector configuration is feasible exactly for

$$
t\in\{-\sqrt2,+\sqrt2\}.
$$

**Proof.**

Let row $0$ be $u$, and rows $496$ and $500$ be $v,w$.  If $D_{ij}(t)$ is their unnormalized dot product and $N_i(t)$ their squared norm, then $D_{ij}(t)>0$ for both pairs and compatibility is equivalent to $N_iN_j-4D_{ij}^2\ge0$.  Direct substitution gives:

- $N_uN_v-4D_{uv}^2=32(t^2-2)$, forcing $t^2\ge2$;
- $N_vN_w-4D_{vw}^2=-4(t^2-2)(3t^2+10)$, forcing $t^2\le2$.

Hence $t^2=2$.

**Verification of sufficiency.** The next cell verifies all $182{,}106$ pairwise inequalities exactly for both $t=\sqrt2$ and $t=-\sqrt2$, proving feasibility. It also performs the exact Gram-matrix calculations used in Proposition 3.2 and Theorem 3.3. $\square$

In [6]:
gram_rational_3, gram_sqrt2_3 = gram_coefficients(P_all[2], Q_all[2])
upper = np.triu(np.ones((604, 604), dtype=bool), 1)
irrational = upper & (gram_sqrt2_3 != 0)
cc = int(np.count_nonzero(irrational[:CORE_N, :CORE_N]))
ca = int(np.count_nonzero(irrational[:CORE_N, CORE_N:]))
aa = int(np.count_nonzero(irrational[CORE_N:, CORE_N:]))
assert (cc, ca, aa, int(np.count_nonzero(irrational))) == (0, 18944, 0, 18944)
print("irrational unordered Gram entries (core-core, cross, extension-extension):", (cc, ca, aa))

minor_relative = [0, 2, 4, 8, 10, 26, 28, 34, 36, 42, 44]
minor_global = [CORE_N + i for i in minor_relative]
G_minor = sp.Matrix([
    [sp.Rational(int(gram_rational_3[i, j]), 4 * DEN**2) for j in minor_global]
    for i in minor_global
])
minor_det = sp.factor(G_minor.det())
assert G_minor.rank() == 11 and minor_det == sp.Rational(1, 512)
print("extension rank witness: rank", G_minor.rank(), "determinant", minor_det)

# The two locking pairs. D(t) and N(t) use numerator coordinates P+Qt;
# for positive D, compatibility is N_i(t)N_j(t)-4D(t)^2 >= 0.
t = sp.symbols("t", real=True)
def symbolic_dot(i, j):
    return sp.expand(sum((int(P_3_denom2[i,k]) + int(Q_3_denom2[i,k])*t) *
                         (int(P_3_denom2[j,k]) + int(Q_3_denom2[j,k])*t) for k in range(11)))
def compatibility_margin(i, j):
    return sp.factor(symbolic_dot(i, i) * symbolic_dot(j, j) - 4 * symbolic_dot(i, j)**2)

lock_lower = compatibility_margin(0, 496)
lock_upper = compatibility_margin(496, 500)
print("row-pair (0,496) margin:", lock_lower)
print("row-pair (496,500) margin:", lock_upper)
assert sp.expand(lock_lower - 32 * (t**2 - 2)) == 0
assert sp.expand(lock_upper + 4 * (t**2 - 2) * (3*t**2 + 10)) == 0

# Both algebraic choices pass all 182,106 pair inequalities exactly.
assert audit_configuration(3 * P_3_denom2, 3 * Q_3_denom2) == expected["3"]
assert audit_configuration(3 * P_3_denom2, -3 * Q_3_denom2) == expected["3"]

# Exhaustively classify the upper-pinning extension pairs.
pinning = Counter()
for i in range(CORE_N, 604):
    for j in range(i + 1, 604):
        D = symbolic_dot(i, j)
        poly = compatibility_margin(i, j)
        if sp.signsimp(D.subs(t, sp.sqrt(2))) > 0 and sp.expand(poly.subs(t, sp.sqrt(2))) == 0 and poly.subs(t, sp.Rational(3,2)) < 0:
            pinning[str(sp.factor(poly))] += 1
assert sorted(pinning.values()) == [72, 612] and sum(pinning.values()) == 684
print("upper-pinning polynomial families:", dict(pinning))


irrational unordered Gram entries (core-core, cross, extension-extension): (0, 18944, 0)


extension rank witness: rank 11 determinant 1/512
row-pair (0,496) margin: 32*(t**2 - 2)
row-pair (496,500) margin: -4*(t**2 - 2)*(3*t**2 + 10)


upper-pinning polynomial families: {'-4*(t**2 - 2)*(3*t**2 + 10)': 612, '-48*t**2*(t**2 - 2)': 72}


**Station Reference.** The exact configuration and its $496+108$ decomposition come from Archives #73 and #74 and Evaluations #1115, #1118, #1125, and #1147. The module decomposition and matching law are from Archives #74 and #102 and Evaluations #1125, #1194, #1198, #1640, #1647, #1652, and #1666. Gram localization, the rational-realization obstruction, and the scalar-family analysis are from Evaluations #1751--#1753, Archives #110 and #116, and Evaluations #1877, #1887, and #1893.

**Related Work.** The classical $582$-point construction arising from [Best (1977)](https://ir.cwi.nl/pub/7433)'s constant-weight code combines coordinate axes with signed weight-four vectors in the norm-four $D_{11}$ shell. Construction 3 follows the related idea of starting from a sparse integer core, but then attaches an algebraic extension over $\mathbb Q(\sqrt2)$. Eleven rotated-frame patterns generate the extension, whose decomposition into rank-five modules and a perfect matching explains $54=6+4\cdot12$.

<a id="section-4"></a>

## 4. S3. Why the classical $D_{11}$ construction stops at $582$.

Let $J_{\pm}(n,4)$ have vertices $(T,\varepsilon)$, where $T\subset[n]$ has size four and $\varepsilon\in\{\pm1\}^T$.  Two vertices conflict when the corresponding signed weight-four vectors have dot product greater than $2$.  Let $A(n,4,4)$ be the largest family of four-subsets with pairwise intersections at most two.

**Theorem 4.1 (signed-shell identity).**

$$
\alpha(J_{\pm}(n,4))=16A(n,4,4).
$$

**Proof.**

Distinct signed weight-four rows conflict exactly when their supports meet in three coordinates and their signs agree on that common triple.  Consider any feasible local sign sets $P_T\subseteq\{\pm1\}^T$, and put $U=\bigcup T$.  Count pairs $(g,T)$ for which $g\in\{\pm1\}^U$ and $g|_T\in P_T$.

Each selected local pattern has $2^{|U|-4}$ global extensions, so the count is

$$
2^{|U|-4}\sum_T|P_T|.
$$

For fixed $g$, the selected supports form a family with intersections at most two; otherwise two restrictions agree on a common triple and conflict.  Therefore the same count is at most $2^{|U|}A(n,4,4)$, giving

$$
\sum_T|P_T|\le16A(n,4,4).
$$

Equality is achieved by taking all sixteen signings on every support in an optimal support family. $\square$

**Corollary 4.2 (dimension eleven).**

The complete norm-four $D_{11}$ shell has maximum compatible subset size $582$.

**Proof.** [Best (1977)](https://ir.cwi.nl/pub/7433) proved that $A(11,4,4)=35$. Theorem 4.1 therefore gives at most $16\cdot35=560$ signed weight-four vectors. The $22$ axes $\{\pm2e_i\}$ are mutually compatible and compatible with every weight-four vector, giving the exact maximum

$$
560+22=582.
$$

$\square$

**Formal proof:** [Lean 4 source](https://github.com/dualverse-ai/station_data_v2/tree/main/artifacts/kissing_number/formal_proofs/s3).

**Verification.** The next cell verifies the explicit $582$-point construction and every geometric inequality.


In [7]:
shell = bundle["shell_582"].astype(np.int64)
assert shell.shape == (582, 11)
G582 = shell @ shell.T
assert np.all(np.diag(G582) == 4)
assert np.max(G582 - 4*np.eye(582, dtype=np.int64)) == 2

axes_mask = np.sum(shell != 0, axis=1) == 1
weight4_mask = np.sum(shell != 0, axis=1) == 4
assert (int(axes_mask.sum()), int(weight4_mask.sum())) == (22, 560)

supports = Counter(tuple(np.flatnonzero(row).tolist()) for row in shell[weight4_mask])
assert len(supports) == 35 and set(supports.values()) == {16}
support_list = sorted(supports)
assert max(len(set(a) & set(b)) for a, b in combinations(support_list, 2)) <= 2
assert len(row_keys(2*shell, np.zeros_like(shell))) == 582
print("verified shell decomposition: 22 axes + 35 supports x 16 signs =", len(shell))
print("maximum support intersection:", max(len(set(a)&set(b)) for a,b in combinations(support_list,2)))


verified shell decomposition: 22 axes + 35 supports x 16 signs = 582
maximum support intersection: 2


**Station Reference.** In S1, the global-sign proof is recorded in Archives #55, #56, and #57 and independently audited in Evaluation #932; the explicit $582$-row witness is the exact shell artifact later packaged as `d11_norm4_optimum.npz`. S2 independently derived the same theorem in Archive #98; Evaluation #2159 supplies its explicit $35$-support witness at $n=11$, and Evaluation #2194 gives exact finite-solver corroboration.

**Related Work.** [Takhanov and Yun (2026)](https://arxiv.org/abs/2606.03299), submitted on June 2, 2026, prove a broader signed-Johnson theorem whose $k=4$ specialization gives $\alpha(J_{\pm}(n,4))=16A(n,4,4)$ and the same $582$ ceiling. The Station derived this specialization independently.

<a id="section-5"></a>

## 5. Additional findings

These are additional results that may be interesting but are not included in the spotlight.


### 5.1 Construction 3 as an oblique shadow of $D_{12}$

We prove that Construction 3 is the image of $604$ norm-four vectors that generate the lattice $D_{12}$ under a single rank-one oblique projection into $\mathbb R^{11}$. This lift replaces the apparent quadratic-coordinate construction by an integral configuration with one reservoir coordinate, while an exact projection identity recovers every norm and inner product. It explains the occurrence of $\sqrt2$ as the effect of coupling the reservoir direction to two visible coordinates and gives a lattice-theoretic construction principle for the $604$-point configuration.

Let

$$
D_{12}=\left\{y\in\mathbb Z^{12}:\sum_{i=1}^{12}y_i\equiv0\pmod2\right\}.
$$

Write $y=(z,r)\in\mathbb R^{11}\times\mathbb R$ and put $c=e_7+e_9$ (one-based indices).  Define

$$
\Pi(z,r)=z+\frac r{\sqrt2}c.
$$

**Lemma 5.1 (rank-one projection identity).**

For $u=(z,r)$ and $v=(w,s)$,

$$
\langle\Pi u,\Pi v\rangle
=\langle u,v\rangle_{\mathbb R^{12}}
+\frac{r(w_7+w_9)+s(z_7+z_9)}{\sqrt2}.
$$

In particular,

$$
\|\Pi(z,r)\|^2=\|(z,r)\|^2+\sqrt2\,r(z_7+z_9),
$$

so norm is preserved when $r(z_7+z_9)=0$.

**Proof.**

Expand $\langle z+rc/\sqrt2,w+sc/\sqrt2\rangle$ and use $\langle c,c\rangle=2$.  The last term is $rs$, exactly the twelfth-coordinate contribution already present in $\langle u,v\rangle$. $\square$

**Proposition 5.2 (exact $D_{12}$ interpretation of Construction 3).**

Construction 3 lifts to $604$ vectors $y\in D_{12}$ of squared norm $4$ that satisfy the projection condition, generate the full lattice $D_{12}$, and project exactly to Construction 3.

**Verification.** The next cell checks the norm, parity, projection condition, and projection identity row by row. Its Smith diagonal is $(1,\ldots,1,2)$, so the row lattice has index two in $\mathbb Z^{12}$. Since it lies in the index-two lattice $D_{12}$, the two lattices are equal. The cell also counts the values of the twelfth coordinate.

In [8]:
from sympy.matrices.normalforms import smith_normal_form
from sympy.polys.domains import ZZ

Y = bundle["d12_lift_3"].astype(np.int64)
assert Y.shape == (604, 12)
assert np.all(np.sum(Y * Y, axis=1) == 4)
assert np.all(np.sum(Y, axis=1) % 2 == 0)
assert np.all(Y[:, 11] * (Y[:, 6] + Y[:, 8]) == 0)

# Reproject into denominator-two coordinates and compare with Construction 3.
projected_P = (2 * Y[:, :11]).astype(np.int64)
projected_Q = np.zeros((604, 11), dtype=np.int64)
projected_Q[:, 6] = Y[:, 11]
projected_Q[:, 8] = Y[:, 11]
assert np.array_equal(projected_P, P_3_denom2)
assert np.array_equal(projected_Q, Q_3_denom2)

# Smith invariants: product 2 means index two in Z^12, hence exactly D12.
S = smith_normal_form(sp.Matrix(Y), domain=ZZ)
smith_diagonal = [abs(int(S[i, i])) for i in range(12)]
assert smith_diagonal == [1] * 11 + [2]

reservoir_histogram = Counter(Y[:, 11].tolist())
assert reservoir_histogram == Counter({0: 530, -1: 36, 1: 36, -2: 1, 2: 1})
print("Smith diagonal:", smith_diagonal)
print("reservoir-coordinate histogram:", dict(sorted(reservoir_histogram.items())))
print("floor/reservoir split: 530 + 74 (different from the 496 + 108 core/extension split)")


Smith diagonal: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2]
reservoir-coordinate histogram: {-2: 1, -1: 36, 0: 530, 1: 36, 2: 1}
floor/reservoir split: 530 + 74 (different from the 496 + 108 core/extension split)


**Station Reference.** The lift and projection identity come from Archives #88, #89, and #114 and Evaluations #1453, #1458, #1460, #1627, and #1629. The $496+108$ construction split and the $530+74$ split by twelfth coordinate are different decompositions.

**Related Work.** The lattice $D_{12}$ and lattice projections are classical. Here the exact rank-one projection realizes Construction 3 as the image of an integral $D_{12}$ configuration and isolates its irrationality in one reservoir direction.

### 5.2 Four maximum compatible selections in a $624$-vector parent shell

We enlarge the integer core of Construction 3 by adjoining the missing antipodes of its $128$ unpaired vectors, producing a centrally symmetric $624$-vector parent shell. We prove that this shell has exactly four maximum compatible subsets of size $496$. One is Construction 3's core, one is the core shared by Constructions 1 and 2, and a third is the negative of Construction 3's core. Global negation exchanges the first and third while preserving the other two, so the four labelled subsets represent at most three isometry classes.

Construction 3's integer core has $184$ antipodal pairs and $128$ singleton rows. Add the missing antipode of every singleton. The resulting set $M$ contains $624$ norm-four integer vectors and is centrally symmetric, but it is not itself a kissing configuration: its conflict edges have dot product $3$.

**Lemma 5.3 (regular bipartite maximum independent sets).**

Let $G=(A\sqcup B,E)$ be a connected $d$-regular bipartite graph with $|A|=|B|=n$. Its independence number is $n$, and its only maximum independent sets are $A$ and $B$.

**Proof.**

If an independent set contains $S\subseteq A$, it can contain at most $n-|N(S)|$ vertices of $B$. Regularity gives $|N(S)|\ge|S|$ by counting the $d|S|$ incident edges, so every independent set has size at most $n$; the sets $A$ and $B$ attain this bound. Equality for a nonempty proper $S$ would mean that every edge incident to $N(S)$ returns to $S$, making $S\cup N(S)$ a union of connected components. Connectedness rules this out. Hence equality occurs only for $S=\varnothing$ or $S=A$. $\square$

**Theorem 5.4 (four maximum selections).**

The parent shell $M$ has exactly four maximum compatible subsets, each of size $496$. Global negation exchanges one pair and fixes the other two setwise, so these subsets represent at most three isometry classes.

**Verification.** The next cell constructs the parent shell and directly bipartitions its two conflict components. It constructs all four maximum selections, checks their sizes and kissing inequalities, identifies the cores of Constructions 1, 2, and 3, and verifies the action of global negation.


In [9]:
# The first 496 rows of Construction 3 are integral.
assert np.all(Q_3_denom2[:CORE_N] == 0) and np.all(P_3_denom2[:CORE_N] % 2 == 0)
core = (P_3_denom2[:CORE_N] // 2).astype(np.int64)
core_keys = {tuple(row.tolist()) for row in core}
singletons = [row.copy() for row in core if tuple((-row).tolist()) not in core_keys]
paired_rows = [row for row in core if tuple((-row).tolist()) in core_keys]
assert len(singletons) == 128 and len(paired_rows) == 368

mother = np.vstack([core, -np.asarray(singletons, dtype=np.int64)])
mother_keys = {tuple(row.tolist()) for row in mother}
assert mother.shape == (624, 11) and len(mother_keys) == 624
assert all(tuple((-row).tolist()) in mother_keys for row in mother)
GM = mother @ mother.T
conflicts = np.transpose(np.nonzero(np.triu(GM > 2, 1)))
assert len(conflicts) == 768 and set(GM[i, j] for i, j in conflicts) == {3}

# The 368 rows already paired in the original core are conflict-isolated.
fixed_indices = {
    i for i, row in enumerate(mother[:CORE_N])
    if tuple((-row).tolist()) in core_keys
}
assert len(fixed_indices) == 368
assert all(int(i) not in fixed_indices and int(j) not in fixed_indices for i, j in conflicts)

# The remaining conflict graph has two connected 6-regular bipartite components.
conflict_adj = [[] for _ in range(624)]
for i, j in conflicts:
    conflict_adj[int(i)].append(int(j))
    conflict_adj[int(j)].append(int(i))

seen = set(fixed_indices)
components = []
for start in range(624):
    if start in seen:
        continue
    component = []
    queue = deque([start])
    seen.add(start)
    while queue:
        i = queue.popleft()
        component.append(i)
        for j in conflict_adj[i]:
            if j not in seen:
                seen.add(j)
                queue.append(j)
    components.append(component)
assert sorted(map(len, components)) == [128, 128]

component_parts = []
for component in components:
    assert {len(conflict_adj[i]) for i in component} == {6}
    color = {component[0]: 0}
    queue = deque([component[0]])
    while queue:
        i = queue.popleft()
        for j in conflict_adj[i]:
            if j not in color:
                color[j] = 1 - color[i]
                queue.append(j)
            else:
                assert color[j] != color[i]
    assert set(color) == set(component) and Counter(color.values()) == Counter({0: 64, 1: 64})
    component_parts.append([
        {i for i, value in color.items() if value == side}
        for side in (0, 1)
    ])

# Lemma 5.3 gives two maximum choices per component. Construct all four.
selections = {}
for choices in product((0, 1), repeat=2):
    indices = set(fixed_indices)
    for component_index, choice in enumerate(choices):
        indices |= component_parts[component_index][choice]
    assert len(indices) == 496
    ordered = sorted(indices)
    selected_gram = GM[np.ix_(ordered, ordered)].copy()
    np.fill_diagonal(selected_gram, 0)
    assert int(selected_gram.max()) <= 2
    selections[choices] = frozenset(tuple(mother[i].tolist()) for i in indices)
assert len(set(selections.values())) == 4

# Identify the named cores and the action of global negation.
def integral_core(label_index):
    assert np.all(Q_all[label_index, :CORE_N] == 0)
    assert np.all(P_all[label_index, :CORE_N] % DEN == 0)
    return frozenset(tuple(row.tolist()) for row in P_all[label_index, :CORE_N] // DEN)

named_cores = {label: integral_core(i) for i, label in enumerate(labels)}
assert named_cores["1"] == named_cores["2"]
selection_names = {}
for choices, rows in selections.items():
    if rows == named_cores["3"]:
        selection_names[choices] = "Construction 3 core"
    elif rows == named_cores["1"]:
        selection_names[choices] = "shared 1/2 core"
    elif rows == frozenset(tuple(-np.asarray(row)) for row in named_cores["3"]):
        selection_names[choices] = "negative Construction 3 core"
    else:
        selection_names[choices] = "fourth selection"
assert Counter(selection_names.values()) == Counter({
    "Construction 3 core": 1, "shared 1/2 core": 1, "negative Construction 3 core": 1, "fourth selection": 1,
})

negation_action = {}
for choices, rows in selections.items():
    negative = frozenset(tuple(-np.asarray(row)) for row in rows)
    targets = [target for target, target_rows in selections.items() if target_rows == negative]
    assert len(targets) == 1
    negation_action[choices] = targets[0]
assert sum(negation_action[key] == key for key in selections) == 2
assert sum(negation_action[key] != key for key in selections) == 2

print("mother shell:", len(mother), "rows; conflicts:", len(conflicts))
print("conflict components:", [len(component) for component in components],
      "with bipartitions", [tuple(map(len, parts)) for parts in component_parts])
print("maximum compatible 496-row selections:", len(selections))
for choices in sorted(selections):
    print(choices, selection_names[choices], "negates to", negation_action[choices])


mother shell: 624 rows; conflicts: 768
conflict components: [128, 128] with bipartitions [(64, 64), (64, 64)]
maximum compatible 496-row selections: 4
(0, 0) Construction 3 core negates to (1, 1)
(0, 1) shared 1/2 core negates to (0, 1)
(1, 0) fourth selection negates to (1, 0)
(1, 1) negative Construction 3 core negates to (0, 0)


**Station Reference.** The parent shell and selection enumeration are from Archive #87, Evaluations #1397, #1406, and #1437, and the corrected synthesis in Question Room discussion #37.

**Related Work.** Maximum independent sets in regular bipartite graphs are classical. Applied to the conflict graph here, this structure yields four maximum compatible $496$-row selections in the $624$-vector parent shell.